# Phase 3 — Behaviour Agent (GRU/LSTM) + KYC/AML Rules Agent**Dev 2 | Branch:** `feat/behaviour-kyc-agents`  **Phase:** 3 of 5 (Detection / Research workstream)This notebook implements and validates two agents defined in AGENTS.md §8.2:| Agent | Type | Primary Model ||---|---|---|| **Behaviour Agent** | Supervised/Unsupervised | GRU (primary), LSTM (comparison) || **KYC/AML Rules Agent** | Deterministic rules | Weighted rule scoring |**Notebook flow:**1. Setup & device detection2. Data loading and preparation3. KYC/AML Rules Agent — fit, predict, evaluate4. Behaviour Agent (GRU) — fit, predict, evaluate5. Behaviour Agent (LSTM) — comparison run6. Side-by-side evaluation metrics7. Save agents8. Conclusion

## 1. Setup

In [ ]:
import sysimport osimport randomimport loggingimport warningswarnings.filterwarnings('ignore')# ── Kaggle: install only missing packages ─────────────────────────────────────try:    import torchexcept ImportError:    import subprocess    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.5.1'], check=False)# ── Project root on sys.path ──────────────────────────────────────────────────for candidate in (    os.path.abspath(os.path.join(os.getcwd(), '..', '..')),    os.path.abspath(os.path.join(os.getcwd(), '..')),    os.getcwd(),):    if os.path.isdir(os.path.join(candidate, 'src')):        if candidate not in sys.path:            sys.path.insert(0, candidate)        break# ── Reproducibility seeds ─────────────────────────────────────────────────────RANDOM_SEED = 42random.seed(RANDOM_SEED)import numpy as npnp.random.seed(RANDOM_SEED)try:    import torch    torch.manual_seed(RANDOM_SEED)    if torch.cuda.is_available():        torch.cuda.manual_seed_all(RANDOM_SEED)    TORCH_AVAILABLE = Trueexcept ImportError:    TORCH_AVAILABLE = False# ── Logging ───────────────────────────────────────────────────────────────────logging.basicConfig(    level=logging.INFO,    format='%(asctime)s | %(name)s | %(levelname)s | %(message)s',    datefmt='%H:%M:%S',)logger = logging.getLogger('phase3')# ── Device detection ──────────────────────────────────────────────────────────try:    from src.utils.device_utils import get_device, log_device_info    device_summary = log_device_info()    logger.info('Compute device: %s', device_summary)except Exception as e:    device_summary = 'device_utils unavailable'    logger.warning('Device detection failed: %s', e)print('Python:', sys.version)print('NumPy:', np.__version__)if TORCH_AVAILABLE:    print('PyTorch:', torch.__version__)print('Compute:', device_summary)

## 2. Data Loading and PreparationWe load from the canonical `data/original/original_data/` files available in this repository.  On Kaggle, point `DATA_ROOT` at `/kaggle/input/<dataset-name>/`.

In [ ]:
%%timeimport pandas as pdfrom pathlib import Path# ── Paths ─────────────────────────────────────────────────────────────────────PROJECT_ROOT = Path(sys.path[0]) if sys.path else Path('.').resolve()DATA_ROOT = PROJECT_ROOT / 'data' / 'original' / 'original_data'# Kaggle overrideKAGGLE_INPUT = Path('/kaggle/input')if KAGGLE_INPUT.exists():    candidates = list(KAGGLE_INPUT.iterdir())    if candidates:        DATA_ROOT = candidates[0]TRANSACTIONS_PATH = DATA_ROOT / 'transactions.csv'ACCOUNTS_PATH     = DATA_ROOT / 'accounts.csv'ML_FEATURES_PATH  = DATA_ROOT / 'ml_features.csv'# ── Load raw files ────────────────────────────────────────────────────────────logger.info('Loading transactions from: %s', TRANSACTIONS_PATH)df_txn = pd.read_csv(TRANSACTIONS_PATH, parse_dates=['Date'])logger.info('Loading accounts from: %s', ACCOUNTS_PATH)df_acc = pd.read_csv(ACCOUNTS_PATH, parse_dates=['opened'])logger.info('Loading ml_features (with labels) from: %s', ML_FEATURES_PATH)df_ml  = pd.read_csv(ML_FEATURES_PATH)print(f'Transactions shape : {df_txn.shape}')print(f'Accounts shape     : {df_acc.shape}')print(f'ML Features shape  : {df_ml.shape}')print(f'Fraud label column : is_suspicious_tx')print(f'Fraud rate         : {df_ml["is_suspicious_tx"].mean():.4%}')

In [ ]:
%%time# ── Build a working dataset aligned to the canonical schema ───────────────────df_work = df_txn.copy()df_work['is_fraud'] = df_ml['is_suspicious_tx'].values# Canonical column aliasesdf_work['transaction_id']     = df_work.index.astype(str)df_work['sender_account_id']  = df_work['Sender_account'].astype(str)df_work['receiver_account_id']= df_work['Receiver_account'].astype(str)df_work['amount_npr']         = df_work['amount_local_npr']df_work['original_currency']  = df_work['Payment_currency'].astype(str)df_work['is_cross_border']    = df_work['cross_border_flag'].fillna(0).astype(int)df_work['timestamp']          = pd.to_datetime(    df_work['Date'].astype(str) + ' ' + df_work['Time'].astype(str),    errors='coerce',)# Map payment type to canonical transaction_typeTYPE_MAP = {    'Cash Deposit'  : 'deposit',    'Cross-border'  : 'remittance_outbound',    'Cheque'        : 'payment',    'ACH'           : 'transfer',    'Credit card'   : 'payment',    'Wire'          : 'transfer',    'Cash Withdrawal': 'withdrawal',}df_work['transaction_type'] = df_work['Payment_type'].map(TYPE_MAP).fillna('payment')# Map transmode code to channelCHANNEL_MAP = {    'A': 'mobile_banking',    'B': 'online_banking',    'E': 'branch',    'F': 'atm',    'J': 'pos',    'P': 'mobile_banking',    'Z': 'branch',}df_work['channel'] = df_work['transmode_code'].map(CHANNEL_MAP).fillna('branch')# Remittance corridor for cross-borderdf_work['remittance_corridor'] = np.where(    df_work['is_cross_border'] == 1,    df_work['Sender_bank_location'].astype(str) + '->Nepal',    None)print('Working dataset shape:', df_work.shape)print('Sample columns:', df_work[['transaction_id', 'sender_account_id', 'amount_npr', 'transaction_type', 'channel']].head())

In [ ]:
# ── Build account feature table for KYC/AML agent ────────────────────────────REFERENCE_DATE = pd.Timestamp('2022-12-31')df_acc_work = df_acc.copy()df_acc_work['account_id']       = df_acc_work['account_id'].astype(str)df_acc_work['account_age_days'] = (REFERENCE_DATE - df_acc_work['opened']).dt.days.fillna(0).astype(int)df_acc_work['kyc_verified']     = 1  # All accounts in this dataset are KYC-verified by defaultdf_acc_work['kyc_risk_grade']   = df_acc_work['risk_grade'].str.lower().fillna('low')df_acc_work['is_pep']           = df_acc_work['pep_flag'].fillna(0).astype(int)df_acc_work['is_sanctioned']    = df_acc_work['sanctions_hit'].fillna(0).astype(int)df_acc_work['is_mule']          = 0print('Account feature table shape:', df_acc_work.shape)print(df_acc_work[['account_id', 'kyc_risk_grade', 'is_pep', 'is_sanctioned', 'account_age_days']].head())

In [ ]:
# ── Join account features onto transactions for KYC agent ────────────────────acc_features = df_acc_work[[    'account_id', 'kyc_verified', 'kyc_risk_grade',    'is_pep', 'is_sanctioned', 'account_age_days',]].copy()df_joined = df_work.merge(    acc_features.rename(columns={'account_id': 'sender_account_id'}),    on='sender_account_id',    how='left',)# Fill missing account features with safe defaultsdf_joined['kyc_verified']     = df_joined['kyc_verified'].fillna(1).astype(int)df_joined['kyc_risk_grade']   = df_joined['kyc_risk_grade'].fillna('low')df_joined['is_pep']           = df_joined['is_pep'].fillna(0).astype(int)df_joined['is_sanctioned']    = df_joined['is_sanctioned'].fillna(0).astype(int)df_joined['account_age_days'] = df_joined['account_age_days'].fillna(9999).astype(int)print('Joined dataset shape:', df_joined.shape)print(f'Fraud rate: {df_joined["is_fraud"].mean():.4%}')print('PEP accounts in transactions:', df_joined['is_pep'].sum())print('Sanctioned accounts in transactions:', df_joined['is_sanctioned'].sum())

In [ ]:
# ── Train / Test split (temporal: last 20% by time as test) ──────────────────df_sorted = df_joined.sort_values('timestamp').reset_index(drop=True)split_idx = int(len(df_sorted) * 0.8)df_train = df_sorted.iloc[:split_idx].copy()df_test  = df_sorted.iloc[split_idx:].copy()print(f'Train size : {len(df_train):,}  | fraud rate: {df_train["is_fraud"].mean():.4%}')print(f'Test  size : {len(df_test):,}   | fraud rate: {df_test["is_fraud"].mean():.4%}')

## 3. KYC/AML Rules AgentDeterministic rule-based scoring for account-level compliance risk.

In [ ]:
%%timefrom src.agents.kyc_aml_agent import KYCAMLAgentfrom src.utils.config import get_configconfig = get_config()kyc_agent = KYCAMLAgent(config=config, logger=logger)# Fit (no-op for rules-based agent)kyc_agent.fit(df_train)# Predict on test setkyc_predictions = kyc_agent.predict(df_test)print('KYC/AML predictions shape:', kyc_predictions.shape)print(kyc_predictions.head(10))print(f'\nAlert rate: {kyc_predictions["alert_flag"].mean():.4%}')print(f'Mean risk score: {kyc_predictions["risk_score"].mean():.4f}')

In [ ]:
# ── Evaluate KYC/AML Agent ────────────────────────────────────────────────────from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, aucy_true = df_test['is_fraud'].valuesy_pred_proba = kyc_predictions['risk_score'].valuesy_pred = kyc_predictions['alert_flag'].valuesprint('KYC/AML Agent Evaluation:')print('=' * 60)print(classification_report(y_true, y_pred, target_names=['Legitimate', 'Suspicious']))print(f'AUC-ROC: {roc_auc_score(y_true, y_pred_proba):.4f}')precision, recall, _ = precision_recall_curve(y_true, y_pred_proba)pr_auc = auc(recall, precision)print(f'AUC-PR : {pr_auc:.4f}')

## 4. Behaviour Agent (GRU)Sequence-based anomaly detection using GRU recurrent neural network.

In [ ]:
%%timefrom src.agents.behaviour_agent import BehaviourAgent# Configure for GRUconfig_gru = get_config().model_dump() if hasattr(get_config(), 'model_dump') else dict(get_config().model_dump())config_gru['behaviour_model_type'] = 'gru'config_gru['behaviour_epochs'] = 10config_gru['behaviour_batch_size'] = 64config_gru['behaviour_seq_len'] = 16config_gru['behaviour_hidden_dim'] = 64behaviour_gru = BehaviourAgent(config=config_gru, logger=logger)# Fit on training databehaviour_gru.fit(df_train)

In [ ]:
%%time# Predict on test setgru_predictions = behaviour_gru.predict(df_test)print('GRU predictions shape:', gru_predictions.shape)print(gru_predictions.head(10))print(f'\nAlert rate: {gru_predictions["alert_flag"].mean():.4%}')print(f'Mean risk score: {gru_predictions["risk_score"].mean():.4f}')

In [ ]:
# ── Evaluate Behaviour Agent (GRU) ────────────────────────────────────────────y_pred_proba_gru = gru_predictions['risk_score'].valuesy_pred_gru = gru_predictions['alert_flag'].valuesprint('Behaviour Agent (GRU) Evaluation:')print('=' * 60)print(classification_report(y_true, y_pred_gru, target_names=['Legitimate', 'Suspicious']))print(f'AUC-ROC: {roc_auc_score(y_true, y_pred_proba_gru):.4f}')precision_gru, recall_gru, _ = precision_recall_curve(y_true, y_pred_proba_gru)pr_auc_gru = auc(recall_gru, precision_gru)print(f'AUC-PR : {pr_auc_gru:.4f}')

## 5. Behaviour Agent (LSTM) — Research ComparisonLSTM as a comparison model to GRU.

In [ ]:
%%time# Configure for LSTMconfig_lstm = config_gru.copy()config_lstm['behaviour_model_type'] = 'lstm'behaviour_lstm = BehaviourAgent(config=config_lstm, logger=logger)# Fit on training databehaviour_lstm.fit(df_train)

In [ ]:
%%time# Predict on test setlstm_predictions = behaviour_lstm.predict(df_test)print('LSTM predictions shape:', lstm_predictions.shape)print(lstm_predictions.head(10))print(f'\nAlert rate: {lstm_predictions["alert_flag"].mean():.4%}')print(f'Mean risk score: {lstm_predictions["risk_score"].mean():.4f}')

In [ ]:
# ── Evaluate Behaviour Agent (LSTM) ───────────────────────────────────────────y_pred_proba_lstm = lstm_predictions['risk_score'].valuesy_pred_lstm = lstm_predictions['alert_flag'].valuesprint('Behaviour Agent (LSTM) Evaluation:')print('=' * 60)print(classification_report(y_true, y_pred_lstm, target_names=['Legitimate', 'Suspicious']))print(f'AUC-ROC: {roc_auc_score(y_true, y_pred_proba_lstm):.4f}')precision_lstm, recall_lstm, _ = precision_recall_curve(y_true, y_pred_proba_lstm)pr_auc_lstm = auc(recall_lstm, precision_lstm)print(f'AUC-PR : {pr_auc_lstm:.4f}')

## 6. Side-by-Side Evaluation Metrics

In [ ]:
# ── Comparative metrics table ─────────────────────────────────────────────────results = pd.DataFrame({    'Agent': ['KYC/AML Rules', 'Behaviour (GRU)', 'Behaviour (LSTM)'],    'AUC-ROC': [        roc_auc_score(y_true, y_pred_proba),        roc_auc_score(y_true, y_pred_proba_gru),        roc_auc_score(y_true, y_pred_proba_lstm),    ],    'AUC-PR': [pr_auc, pr_auc_gru, pr_auc_lstm],    'Alert Rate': [        kyc_predictions['alert_flag'].mean(),        gru_predictions['alert_flag'].mean(),        lstm_predictions['alert_flag'].mean(),    ],})print('\n' + '=' * 70)print('PHASE 3 AGENT COMPARISON')print('=' * 70)print(results.to_string(index=False))print('=' * 70)

## 7. Save Trained Agents

In [ ]:
import joblibfrom pathlib import Path# Create models directoryMODELS_DIR = PROJECT_ROOT / 'models'MODELS_DIR.mkdir(exist_ok=True)# Save agentskyc_path = MODELS_DIR / 'kyc_aml_agent.pkl'gru_path = MODELS_DIR / 'behaviour_agent_gru.pkl'lstm_path = MODELS_DIR / 'behaviour_agent_lstm.pkl'joblib.dump(kyc_agent, kyc_path)joblib.dump(behaviour_gru, gru_path)joblib.dump(behaviour_lstm, lstm_path)print(f'Saved KYC/AML agent to: {kyc_path}')print(f'Saved Behaviour (GRU) agent to: {gru_path}')print(f'Saved Behaviour (LSTM) agent to: {lstm_path}')

## 8. Conclusion**Phase 3 complete.**  Both agents have been implemented, trained, and evaluated:1. **KYC/AML Rules Agent** — deterministic, rule-based scoring with configurable weights for NRB compliance rules (PEP, sanctions, structuring, layering).2. **Behaviour Agent (GRU)** — primary sequence model for account-level behaviour anomaly detection.3. **Behaviour Agent (LSTM)** — research comparison model.**Key Findings:**- All agents produce canonical output schema (transaction_id, risk_score, alert_flag, reason_code, explanation, timestamp).- Both agents handle missing features gracefully with fallback strategies.- The agents are serializable (saved via joblib) and can be loaded without refitting.- GRU vs. LSTM performance comparison is complete.**Next Steps:**- Proceed to **Phase 4**: Graph agent (GraphSAGE/GAT) + Meta-learner- Combine all agent outputs in the meta-learner for ensemble fraud detection.